# Actividad 17 — Reentrenamiento Final con Dataset Reconstruido

Dataset: `master_reconstruido_completo.csv` (80 meses, fuentes crudas, normalizacion unica)

| Modelo | Arquitectura | OUT_DIR |
|--------|-------------|---------|
| GE sin NLP | DualLSTM-BahdanauAttention (64/64) | `resultados/ge_final/` |
| GM v3 con NLP | DualLSTM + PCA + Dropout NLP=0.5 | `resultados/gm_v3_final/` |
| XGBoost | Grid search TimeSeriesSplit | `resultados/xgboost_final/` |

In [1]:
import os, json, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from itertools import product
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data/processed/master_reconstruido_completo.csv'
TIMESTEPS = 6; LSTM_UNITS = 64; ATTN_UNITS = 64; DROPOUT = 0.30
L2_REG = 0.001; LR = 1e-3; EPOCHS = 300; BATCH = 8; PATIENCE = 15; N_TEST = 12
TARGET = 'produccion_t'

print(f'TF {tf.__version__} | XGB {xgb.__version__} | Data: {DATA_PATH.exists()}')

TF 2.21.0 | XGB 3.2.0 | Data: True


In [2]:
# --- Carga y preparacion comun ---
df_all = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
df_all = df_all.sort_values('fecha_evento').reset_index(drop=True)

df_all['lag_1'] = df_all[TARGET].shift(1)
df_all['lag_3'] = df_all[TARGET].shift(3)
df_all['lag_6'] = df_all[TARGET].shift(6)
df_all = df_all.dropna().reset_index(drop=True)

n_total = len(df_all); n_test = N_TEST; n_train = n_total - n_test

STRUCT_COLS = [
    'lag_1', 'lag_3', 'lag_6', 'precio_chacra_kg',
    'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas',
    'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M',
    'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon',
    'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos',
]
NLP_COLS = ['nlp_index', 'nlp_index_lag1']

print(f'Dataset con lags: {df_all.shape}  n_train={n_train} n_test={n_test}')
print(f'Train: {df_all.iloc[0].fecha_evento.date()} -> {df_all.iloc[n_train-1].fecha_evento.date()}')
print(f'Test:  {df_all.iloc[n_train].fecha_evento.date()} -> {df_all.iloc[-1].fecha_evento.date()}')

# --- Funciones compartidas ---
class BahdanauAttention(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W_q = layers.Dense(units, use_bias=False)
        self.W_v = layers.Dense(units, use_bias=False)
        self.V   = layers.Dense(1, use_bias=False)
    def call(self, query, values):
        q = tf.expand_dims(self.W_q(query), 1)
        e = self.V(tf.nn.tanh(self.W_v(values) + q))
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(a * values, axis=1), a
    def get_config(self):
        return {**super().get_config(), 'units': self.units}

def make_dual_seq(y, X, sl):
    Xa, Xb, Y = [], [], []
    for i in range(len(y) - sl):
        Xa.append(y[i:i+sl].reshape(-1,1)); Xb.append(X[i:i+sl]); Y.append(y[i+sl])
    return np.array(Xa, dtype=np.float32), np.array(Xb, dtype=np.float32), np.array(Y, dtype=np.float32)

def make_seq(Xs, Xn, y, ts):
    a,b,c = [],[],[]
    for i in range(ts, len(Xs)):
        a.append(Xs[i-ts:i]); b.append(Xn[i-ts:i]); c.append(y[i])
    return np.array(a), np.array(b), np.array(c)

def shock_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    pct = np.abs(np.diff(y_true, prepend=y_true[0]) / (np.abs(y_true) + 1e-8))
    idx_s = np.where(pct > 0.20)[0]
    idx_n = np.where(pct <= 0.20)[0]
    mae_s = float(mean_absolute_error(y_true[idx_s], y_pred[idx_s])) if len(idx_s) > 0 else float('nan')
    mae_n = float(mean_absolute_error(y_true[idx_n], y_pred[idx_n])) if len(idx_n) > 0 else float('nan')
    det = (mae_s - mae) / mae * 100 if len(idx_s) > 0 else float('nan')
    return {'MAE': mae, 'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'R2': float(r2_score(y_true, y_pred)),
            'n_shocks': len(idx_s), 'MAE_shock': mae_s, 'MAE_normal': mae_n, 'deterioro_pct': det}

def save(out_dir, name, m, fechas, y_t, y_p):
    out_dir.mkdir(parents=True, exist_ok=True)
    m['modelo'] = name
    with open(out_dir / 'metricas.json', 'w') as f: json.dump(m, f, indent=2)
    pd.DataFrame({'fecha': fechas, 'real': y_t, 'predicho': y_p}).to_csv(out_dir / 'predicciones.csv', index=False)

print('Funciones listas.')

Dataset con lags: (74, 29)  n_train=62 n_test=12
Train: 2019-07-01 -> 2024-08-01
Test:  2024-09-01 -> 2025-08-01
Funciones listas.


## MODELO 1 — GE sin NLP

In [3]:
# ===== MODELO 1: GE sin NLP =====
tf.random.set_seed(SEED); np.random.seed(SEED)
OUT_GE = ROOT / 'resultados/ge_final'

df_tr = df_all.iloc[:n_train]; df_te = df_all.iloc[n_train:]
y_tr_raw = df_tr[TARGET].values; y_te_raw = df_te[TARGET].values
B_tr_raw = df_tr[STRUCT_COLS].values; B_te_raw = df_te[STRUCT_COLS].values

sc_a = StandardScaler(); sc_b = StandardScaler()
y_tr_sc = sc_a.fit_transform(y_tr_raw.reshape(-1,1)).flatten()
y_te_sc = sc_a.transform(y_te_raw.reshape(-1,1)).flatten()
B_tr_sc = sc_b.fit_transform(B_tr_raw).astype(np.float32)
B_te_sc = sc_b.transform(B_te_raw).astype(np.float32)

i1, i3, i6 = STRUCT_COLS.index('lag_1'), STRUCT_COLS.index('lag_3'), STRUCT_COLS.index('lag_6')

Xa_tr, Xb_tr, y_seq = make_dual_seq(y_tr_sc, B_tr_sc, TIMESTEPS)
ns = len(Xa_tr); nv = max(3, int(ns*0.15)); nt = ns - nv

reg = regularizers.l2(L2_REG)
inp_a = keras.Input(shape=(TIMESTEPS,1), name='ca')
h_a = layers.LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=reg, recurrent_regularizer=reg)(inp_a)
h_a = layers.Dropout(DROPOUT)(h_a)
ctx_a, _ = BahdanauAttention(ATTN_UNITS, name='attn_a')(h_a[:,-1,:], h_a)

inp_b = keras.Input(shape=(TIMESTEPS,len(STRUCT_COLS)), name='cb')
h_b = layers.LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=reg, recurrent_regularizer=reg)(inp_b)
h_b = layers.Dropout(DROPOUT)(h_b)
ctx_b, _ = BahdanauAttention(ATTN_UNITS, name='attn_b')(h_b[:,-1,:], h_b)

mg = layers.Concatenate()([ctx_a, ctx_b])
x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(mg)
x = layers.Dropout(DROPOUT/2)(x)
x = layers.Dense(16, activation='relu')(x)
out = layers.Dense(1)(x)
model_ge = keras.Model([inp_a, inp_b], out, name='GE_final')
model_ge.compile(optimizer=keras.optimizers.Adam(LR), loss='mse', metrics=['mae'])
print(f'GE params: {model_ge.count_params():,}')

hist = model_ge.fit([Xa_tr[:nt], Xb_tr[:nt]], y_seq[:nt],
                    validation_data=([Xa_tr[nt:], Xb_tr[nt:]], y_seq[nt:]),
                    epochs=EPOCHS, batch_size=BATCH, shuffle=False, verbose=0,
                    callbacks=[EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
                               ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-6)])
print(f'Epocas: {len(hist.history["loss"])}  best_val: {min(hist.history["val_loss"]):.6f}')

# Prediccion recursiva
win_a = y_tr_sc[-TIMESTEPS:].copy(); buf = list(y_tr_sc[-6:])
preds = []
for step in range(n_test):
    xa = win_a.reshape(1,TIMESTEPS,1).astype(np.float32)
    s = max(0, step-TIMESTEPS+1); bav = B_te_sc[s:step+1]
    if len(bav) < TIMESTEPS:
        bw = np.vstack([np.tile(B_tr_sc[-1], (TIMESTEPS-len(bav),1)), bav])
    else:
        bw = bav.copy()
    for t in range(TIMESTEPS):
        for idx, off in [(i1,1),(i3,3),(i6,6)]:
            pos = len(buf)-off; bw[t,idx] = buf[pos] if pos >= 0 else 0.0
    xb = bw.reshape(1,TIMESTEPS,len(STRUCT_COLS)).astype(np.float32)
    p = float(model_ge.predict([xa,xb], verbose=0)[0,0])
    preds.append(p); buf.append(p)
    win_a = np.roll(win_a,-1); win_a[-1] = p

y_pred_ge = sc_a.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
m_ge = shock_metrics(y_te_raw, y_pred_ge)
fechas_ge = df_te['fecha_evento'].dt.strftime('%Y-%m-%d').values
save(OUT_GE, 'GE_final', m_ge, fechas_ge, y_te_raw, y_pred_ge)
print(f'\nGE: MAE={m_ge["MAE"]:.4f} RMSE={m_ge["RMSE"]:.4f} R2={m_ge["R2"]:.4f} shocks={m_ge["n_shocks"]} det={m_ge["deterioro_pct"]:+.1f}%')

GE params: 65,249


Epoch 19: early stopping


Restoring model weights from the end of the best epoch: 4.


Epocas: 19  best_val: 0.822255



GE: MAE=0.8833 RMSE=0.9647 R2=-0.1635 shocks=4 det=-4.4%


## MODELO 2 — GM v3 con NLP

In [4]:
# ===== MODELO 2: GM v3 con NLP =====
tf.random.set_seed(SEED); np.random.seed(SEED)
OUT_GM = ROOT / 'resultados/gm_v3_final'

GM_STRUCT = [c for c in df_all.columns if c not in
             ['fecha_evento', TARGET, 'nlp_index', 'nlp_index_lag1',
              'tiene_nlp', 'es_shock', 'lag_1', 'lag_3', 'lag_6']]
GM_STRUCT = GM_STRUCT + ['tiene_nlp']

df_tr_gm = df_all.iloc[:n_train]; df_te_gm = df_all.iloc[n_train:]

sc_s = StandardScaler(); sc_n = StandardScaler(); sc_y = StandardScaler()
Xs_tr = sc_s.fit_transform(df_tr_gm[GM_STRUCT])
Xs_te = sc_s.transform(df_te_gm[GM_STRUCT])
Xn_tr = sc_n.fit_transform(df_tr_gm[NLP_COLS])
Xn_te = sc_n.transform(df_te_gm[NLP_COLS])
y_tr_gm = sc_y.fit_transform(df_tr_gm[[TARGET]])
y_te_gm = sc_y.transform(df_te_gm[[TARGET]])

pca = PCA(n_components=0.95, random_state=SEED)
Xs_tr_p = pca.fit_transform(Xs_tr); Xs_te_p = pca.transform(Xs_te)
print(f'PCA: {Xs_tr.shape[1]} -> {pca.n_components_} componentes')

Xs_sq_tr, Xn_sq_tr, y_sq_tr = make_seq(Xs_tr_p, Xn_tr, y_tr_gm, TIMESTEPS)
Xs_sq_te, Xn_sq_te, y_sq_te = make_seq(Xs_te_p, Xn_te, y_te_gm, TIMESTEPS)
print(f'Seq train: {Xs_sq_tr.shape}  test: {Xs_sq_te.shape}')

def build_gm(ss, ns):
    inp_s = layers.Input(shape=ss)
    h = layers.LSTM(64, return_sequences=True)(inp_s)
    sc = layers.Dense(1, activation='tanh')(h)
    sw = layers.Softmax(axis=1)(sc)
    ca = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(layers.Multiply()([h, sw]))
    ca = layers.Dropout(0.2)(ca)
    inp_n = layers.Input(shape=ns)
    cb = layers.Dropout(0.5, name='drop_nlp')(layers.LSTM(16)(inp_n))
    mg = layers.Concatenate()([ca, cb])
    x = layers.Dropout(0.2)(layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(mg))
    return keras.Model([inp_s, inp_n], layers.Dense(1)(layers.Dense(16, activation='relu')(x)), name='GM_v3_final')

model_gm = build_gm((Xs_sq_tr.shape[1], Xs_sq_tr.shape[2]), (Xn_sq_tr.shape[1], Xn_sq_tr.shape[2]))
model_gm.compile(optimizer=keras.optimizers.Adam(LR), loss='mse', metrics=['mae'])
print(f'GM params: {model_gm.count_params():,}')

hist_gm = model_gm.fit([Xs_sq_tr, Xn_sq_tr], y_sq_tr,
                        epochs=200, batch_size=BATCH, validation_split=0.2,
                        shuffle=False, verbose=0,
                        callbacks=[EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
                                   ReduceLROnPlateau('val_loss', factor=0.5, patience=7, min_lr=1e-6)])
print(f'Epocas: {len(hist_gm.history["loss"])}  best_val: {min(hist_gm.history["val_loss"]):.6f}')

yp_gm = sc_y.inverse_transform(model_gm.predict([Xs_sq_te, Xn_sq_te], verbose=0)).flatten()
yt_gm = sc_y.inverse_transform(y_sq_te).flatten()

m_gm = shock_metrics(yt_gm, yp_gm)
fechas_gm = df_te_gm.iloc[TIMESTEPS:]['fecha_evento'].dt.strftime('%Y-%m-%d').values
save(OUT_GM, 'GM_v3_final', m_gm, fechas_gm, yt_gm, yp_gm)
print(f'\nGM v3: MAE={m_gm["MAE"]:.4f} RMSE={m_gm["RMSE"]:.4f} R2={m_gm["R2"]:.4f} shocks={m_gm["n_shocks"]} det={m_gm["deterioro_pct"]:+.1f}%')

PCA: 21 -> 8 componentes
Seq train: (56, 6, 8)  test: (6, 6, 8)
GM params: 23,106


Epoch 32: early stopping


Restoring model weights from the end of the best epoch: 17.


Epocas: 32  best_val: 1.020671



GM v3: MAE=0.6353 RMSE=0.7989 R2=-7.2581 shocks=1 det=+49.4%


## MODELO 3 — XGBoost

In [5]:
# ===== MODELO 3: XGBoost =====
np.random.seed(SEED)
OUT_XGB = ROOT / 'resultados/xgboost_final'

df_xgb = df_all.copy()
df_xgb['prod_lag2'] = df_xgb[TARGET].shift(2)
df_xgb['prod_roll3_mean'] = df_xgb[TARGET].shift(1).rolling(3).mean()
df_xgb['prod_roll6_mean'] = df_xgb[TARGET].shift(1).rolling(6).mean()
df_xgb['prod_roll3_std']  = df_xgb[TARGET].shift(1).rolling(3).std()
df_xgb = df_xgb.dropna().reset_index(drop=True)

XGB_FEAT = [c for c in df_xgb.columns if c not in ['fecha_evento', TARGET, 'es_shock']]
nt_x = len(df_xgb) - N_TEST

X_tr = df_xgb.iloc[:nt_x][XGB_FEAT].values; y_tr_x = df_xgb.iloc[:nt_x][TARGET].values
X_te = df_xgb.iloc[nt_x:][XGB_FEAT].values; y_te_x = df_xgb.iloc[nt_x:][TARGET].values

print(f'XGBoost: train={nt_x} test={N_TEST} features={len(XGB_FEAT)}')

pgrid = {'max_depth': [2,3,4], 'n_estimators': [50,100,200],
         'learning_rate': [0.05,0.1,0.2], 'subsample': [0.8,1.0], 'colsample_bytree': [0.8,1.0]}
tscv = TimeSeriesSplit(n_splits=3)
best_mae, best_p = float('inf'), {}
combos = list(product(pgrid['max_depth'], pgrid['n_estimators'], pgrid['learning_rate'],
                      pgrid['subsample'], pgrid['colsample_bytree']))
print(f'Grid search: {len(combos)} combinaciones...')

for md, ne, lr, ss, cb in combos:
    maes = []
    for ti, vi in tscv.split(X_tr):
        m = xgb.XGBRegressor(max_depth=md, n_estimators=ne, learning_rate=lr,
                             subsample=ss, colsample_bytree=cb, random_state=SEED, verbosity=0)
        m.fit(X_tr[ti], y_tr_x[ti])
        maes.append(mean_absolute_error(y_tr_x[vi], m.predict(X_tr[vi])))
    cv = np.mean(maes)
    if cv < best_mae:
        best_mae = cv
        best_p = {'max_depth':md, 'n_estimators':ne, 'learning_rate':lr, 'subsample':ss, 'colsample_bytree':cb}

print(f'Mejor CV MAE: {best_mae:.4f}  params: {best_p}')

model_xgb = xgb.XGBRegressor(**best_p, random_state=SEED, verbosity=0)
model_xgb.fit(X_tr, y_tr_x)
yp_xgb = model_xgb.predict(X_te)

m_xgb = shock_metrics(y_te_x, yp_xgb)
m_xgb['best_params'] = best_p
fechas_xgb = df_xgb.iloc[nt_x:]['fecha_evento'].dt.strftime('%Y-%m-%d').values
save(OUT_XGB, 'XGBoost_final', m_xgb, fechas_xgb, y_te_x, yp_xgb)
print(f'\nXGBoost: MAE={m_xgb["MAE"]:.4f} RMSE={m_xgb["RMSE"]:.4f} R2={m_xgb["R2"]:.4f} shocks={m_xgb["n_shocks"]} det={m_xgb["deterioro_pct"]:+.1f}%')

XGBoost: train=56 test=12 features=30
Grid search: 108 combinaciones...


Mejor CV MAE: 0.6009  params: {'max_depth': 2, 'n_estimators': 50, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}

XGBoost: MAE=0.6253 RMSE=0.7579 R2=0.2820 shocks=4 det=+30.8%


---
## RANKING FINAL

In [6]:
orig = {
    'GE sin NLP': {'MAE': 0.0673, 'det': '+2.3%'},
    'GM v3 NLP':  {'MAE': 0.0645, 'det': '+11.7%'},
    'XGBoost':    {'MAE': 0.0471, 'det': '+16.5%'},
}
ext = {
    'GE sin NLP': m_ge,
    'GM v3 NLP':  m_gm,
    'XGBoost':    m_xgb,
}

print('='*72)
print('  RANKING FINAL -- n_train=44 vs n_train=68')
print('='*72)
print(f'  {"Modelo":<14} {"MAE_44":>8} {"MAE_68":>8} {"Delta":>8} {"Det_44":>10} {"Det_68":>10}')
print('-'*72)

for name in ['GE sin NLP', 'GM v3 NLP', 'XGBoost']:
    mae44 = orig[name]['MAE']
    mae68 = ext[name]['MAE']
    delta = (mae68 - mae44) / mae44 * 100
    det44 = orig[name]['det']
    det68_v = ext[name].get('deterioro_pct', float('nan'))
    det68 = f'{det68_v:+.1f}%' if not np.isnan(det68_v) else 'N/A'
    print(f'  {name:<14} {mae44:>8.4f} {mae68:>8.4f} {delta:>+7.1f}% {det44:>10} {det68:>10}')

print('='*72)

# Detalle de shocks
print(f'\n  DETALLE SHOCKS EN TEST')
print(f'  {"Modelo":<14} {"n_shock":>8} {"MAE_glob":>10} {"MAE_shock":>10} {"MAE_norm":>10}')
print('-'*58)
for name, m in [('GE sin NLP', m_ge), ('GM v3 NLP', m_gm), ('XGBoost', m_xgb)]:
    ns = m['n_shocks']
    ms = f'{m["MAE_shock"]:.4f}' if not np.isnan(m['MAE_shock']) else 'N/A'
    mn = f'{m["MAE_normal"]:.4f}' if not np.isnan(m['MAE_normal']) else 'N/A'
    print(f'  {name:<14} {ns:>8} {m["MAE"]:>10.4f} {ms:>10} {mn:>10}')
print('='*58)

# Guardar comparativa
comp = {
    'dataset': 'master_reconstruido_completo.csv',
    'fuente': 'MIDAGRI+NASA+INDECI crudos, normalizacion unica',
    'n_train_orig': 44, 'n_train_ext': n_train, 'n_test': n_test,
    'modelos': {}
}
for name, key_ext in [('GE_sin_NLP', 'GE sin NLP'), ('GM_v3_NLP', 'GM v3 NLP'), ('XGBoost', 'XGBoost')]:
    comp['modelos'][name] = {
        'MAE_44': orig[key_ext]['MAE'], 'det_44': orig[key_ext]['det'],
        'MAE_68': ext[key_ext]['MAE'], 'RMSE_68': ext[key_ext]['RMSE'], 'R2_68': ext[key_ext]['R2'],
        'n_shocks': ext[key_ext]['n_shocks'],
        'MAE_shock': ext[key_ext]['MAE_shock'] if not np.isnan(ext[key_ext]['MAE_shock']) else None,
        'deterioro_68': ext[key_ext]['deterioro_pct'] if not np.isnan(ext[key_ext]['deterioro_pct']) else None,
    }
with open(ROOT / 'resultados/ranking_final_44vs68.json', 'w') as f:
    json.dump(comp, f, indent=2)
print(f'\nGuardado: resultados/ranking_final_44vs68.json')

  RANKING FINAL -- n_train=44 vs n_train=68
  Modelo           MAE_44   MAE_68    Delta     Det_44     Det_68
------------------------------------------------------------------------
  GE sin NLP       0.0673   0.8833 +1212.5%      +2.3%      -4.4%
  GM v3 NLP        0.0645   0.6353  +884.9%     +11.7%     +49.4%
  XGBoost          0.0471   0.6253 +1227.7%     +16.5%     +30.8%

  DETALLE SHOCKS EN TEST
  Modelo          n_shock   MAE_glob  MAE_shock   MAE_norm
----------------------------------------------------------
  GE sin NLP            4     0.8833     0.8444     0.9027
  GM v3 NLP             1     0.6353     0.9493     0.5725
  XGBoost               4     0.6253     0.8178     0.5291

Guardado: resultados/ranking_final_44vs68.json
